# Simulation Robustness Testing

This notebook evaluates how robust the DoH detection classifier is under controlled flow-feature perturbations. Instead of generating new packet captures or modifying raw packets directly, this notebook focuses on feature-level simulation. That means we take the same flow features used by the detector and apply controlled changes to timing, packet size, and packet rate features.

The goal is to test whether small, realistic changes in traffic behavior can reduce the model’s ability to detect malicious DoH traffic. This helps measure how sensitive the classifier is to distribution shifts and whether flow-based detection remains reliable when malicious traffic is slightly altered to look more like benign traffic.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

np.random.seed(42)

DATA_PATH = "L2-BenignDoH-MaliciousDoH.parquet"
MODEL_PATH = "detector_model.joblib"  # update once model is available
LABEL_COL = "Label"

## Load and Prepare Data

We load the labeled flow-feature dataset, remove invalid values, drop identifier columns, and separate features from labels.

In [2]:
df = pd.read_parquet(DATA_PATH)

print("Original dataset shape:", df.shape)
print(df[LABEL_COL].value_counts())

drop_cols = [
    "src_ip", "dst_ip", "src_mac", "dst_mac",
    "src_port", "dst_port", "timestamp", "flow_id"
]

df = df.replace([np.inf, -np.inf, "Infinity", "-Infinity"], np.nan)
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors="ignore")

df = df.dropna(subset=[LABEL_COL])

feature_cols = [col for col in df.columns if col != LABEL_COL]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[feature_cols] = df[feature_cols].fillna(0)

X = df.drop(columns=[LABEL_COL])
y = df[LABEL_COL]

print("Cleaned dataset shape:", df.shape)
print("Feature shape:", X.shape)
print(y.value_counts())

X.head()

Original dataset shape: (268661, 30)
Label
Malicious    249538
Benign        19123
Name: count, dtype: int64
Cleaned dataset shape: (268661, 30)
Feature shape: (268661, 29)
Label
Malicious    249538
Benign        19123
Name: count, dtype: int64


,Duration,FlowBytesSent,FlowSentRate,FlowBytesReceived,FlowReceivedRate,PacketLengthVariance,PacketLengthStandardDeviation,PacketLengthMean,PacketLengthMedian,PacketLengthMode,...,PacketTimeSkewFromMode,PacketTimeCoefficientofVariation,ResponseTimeTimeVariance,ResponseTimeTimeStandardDeviation,ResponseTimeTimeMean,ResponseTimeTimeMedian,ResponseTimeTimeMode,ResponseTimeTimeSkewFromMedian,ResponseTimeTimeSkewFromMode,ResponseTimeTimeCoefficientofVariation
0,95.081551,62311,655.342712,65358,687.388855,7474.676758,86.456215,135.673752,102.0,54,...,1.682529,0.574626,0.001053,0.032457,0.027624,0.026854,0.026822,0.071187,0.024715,1.174948
1,122.309319,93828,767.136963,101232,827.671997,10458.118164,102.264946,141.245468,114.0,54,...,0.772748,0.509047,0.001170,0.034200,0.024387,0.021043,0.026981,0.293297,-0.075845,1.402382
2,120.958412,38784,320.639130,38236,316.108643,7300.293945,85.441757,133.715271,89.0,54,...,1.353607,0.732636,0.000785,0.028021,0.029238,0.026921,0.026855,0.248064,0.085061,0.958348
3,110.501083,61993,561.017151,69757,631.278870,8499.282227,92.191551,139.123550,114.0,54,...,1.148758,0.646859,0.000411,0.020274,0.019925,0.019268,0.026918,0.097199,-0.344926,1.017535
4,54.229893,83641,1542.341309,76804,1416.266968,8052.745605,89.737091,138.913422,114.0,114,...,1.573873,0.507334,0.079079,0.281209,0.025930,0.000047,0.000021,0.276133,0.092135,10.844830


## Load Model and Evaluation Helper

The trained classifier is loaded from a serialized `.joblib` file. The evaluation helper reports core performance metrics for each simulation scenario, allowing us to compare how performance changes under different traffic perturbations.

NOTE: Replace MODEL_PATH with the actual file name once received

In [3]:
try:
    model = joblib.load(MODEL_PATH)
    print(f"Loaded model from {MODEL_PATH}")
except FileNotFoundError:
    model = None
    print(f"Model not found at {MODEL_PATH}. Add the trained .joblib file before running evaluation.")

# Temporary fallback model for testing notebook flow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

if model is None:
    print("No saved model found. Training temporary Random Forest model for testing notebook flow.")

    class_counts = y.value_counts()
    print("Class counts:")
    print(class_counts)

    can_stratify = class_counts.min() >= 2

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y if can_stratify else None
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)

    X = X_test
    y = y_test

    print("Temporary model trained successfully.")

def evaluate(model, X_eval, y_eval, scenario):
    if model is None:
        return {
            "scenario": scenario,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "note": "model not loaded"
        }

    preds = model.predict(X_eval)

    return {
        "scenario": scenario,
        "accuracy": accuracy_score(y_eval, preds),
        "precision": precision_score(y_eval, preds, pos_label="Malicious", zero_division=0),
        "recall": recall_score(y_eval, preds, pos_label="Malicious", zero_division=0),
        "f1": f1_score(y_eval, preds, pos_label="Malicious", zero_division=0),
        "false_negatives": int(((y_eval == "Malicious") & (preds == "Benign")).sum())
    }

Model not found at detector_model.joblib. Add the trained .joblib file before running evaluation.
No saved model found. Training temporary Random Forest model for testing notebook flow.
Class counts:
Label
Malicious    249538
Benign        19123
Name: count, dtype: int64
Temporary model trained successfully.


## Perturbation Experiments

We define controlled feature-level simulations that modify timing, packet size, and packet rate features. These represent realistic changes an attacker could make to reduce the distinctiveness of malicious DoH traffic.

In [4]:
def perturb_columns(X_input, keywords, mode="noise", noise_level=0.2, scale_factor=1.25):
    X_mod = X_input.copy()

    selected_cols = [
        col for col in X_mod.columns
        if any(keyword in col.lower() for keyword in keywords)
    ]

    for col in selected_cols:
        if mode == "noise":
            noise = np.random.normal(loc=1.0, scale=noise_level, size=len(X_mod))
            noise = np.clip(noise, 0.1, None)
            X_mod[col] = X_mod[col] * noise
        elif mode == "scale":
            X_mod[col] = X_mod[col] * scale_factor

    return X_mod, selected_cols


experiments = {}

experiments["Original Traffic"] = X

experiments["Timing Perturbation"], timing_cols = perturb_columns(
    X,
    keywords=["iat", "duration", "time"],
    mode="noise",
    noise_level=0.20
)

experiments["Packet Size Perturbation"], size_cols = perturb_columns(
    X,
    keywords=["len", "byte", "byts", "totlen", "pkt_size"],
    mode="scale",
    scale_factor=1.25
)

experiments["Packet Rate Perturbation"], rate_cols = perturb_columns(
    X,
    keywords=["pkts_s", "pkt_s", "rate"],
    mode="noise",
    noise_level=0.30
)

X_combined = X.copy()
X_combined, _ = perturb_columns(X_combined, ["iat", "duration", "time"], mode="noise", noise_level=0.20)
X_combined, _ = perturb_columns(X_combined, ["len", "byte", "byts", "totlen", "pkt_size"], mode="scale", scale_factor=1.25)
X_combined, _ = perturb_columns(X_combined, ["pkts_s", "pkt_s", "rate"], mode="noise", noise_level=0.30)

experiments["Combined Perturbation"] = X_combined

print("Timing columns:", timing_cols)
print("Packet size columns:", size_cols)
print("Packet rate columns:", rate_cols)

Timing columns: ['Duration', 'PacketLengthStandardDeviation', 'PacketLengthCoefficientofVariation', 'PacketTimeVariance', 'PacketTimeStandardDeviation', 'PacketTimeMean', 'PacketTimeMedian', 'PacketTimeMode', 'PacketTimeSkewFromMedian', 'PacketTimeSkewFromMode', 'PacketTimeCoefficientofVariation', 'ResponseTimeTimeVariance', 'ResponseTimeTimeStandardDeviation', 'ResponseTimeTimeMean', 'ResponseTimeTimeMedian', 'ResponseTimeTimeMode', 'ResponseTimeTimeSkewFromMedian', 'ResponseTimeTimeSkewFromMode', 'ResponseTimeTimeCoefficientofVariation']
Packet size columns: ['FlowBytesSent', 'FlowBytesReceived', 'PacketLengthVariance', 'PacketLengthStandardDeviation', 'PacketLengthMean', 'PacketLengthMedian', 'PacketLengthMode', 'PacketLengthSkewFromMedian', 'PacketLengthSkewFromMode', 'PacketLengthCoefficientofVariation']
Packet rate columns: ['FlowSentRate', 'FlowReceivedRate']


## Robustness Evaluation

Each perturbation is evaluated against the same labels to measure how much classifier performance changes under simulated traffic variation.

In [5]:
results = []

for scenario, X_eval in experiments.items():
    results.append(evaluate(model, X_eval, y, scenario))

results_df = pd.DataFrame(results)
results_df

,scenario,accuracy,precision,recall,f1,false_negatives
0,Original Traffic,0.999944,0.999980,0.999960,0.999970,2
1,Timing Perturbation,0.998530,0.999880,0.998537,0.999208,73
2,Packet Size Perturbation,0.957252,0.966126,0.988639,0.977253,567
3,Packet Rate Perturbation,0.999926,0.999980,0.999940,0.999960,3
4,Combined Perturbation,0.926191,0.959549,0.961048,0.960298,1944


In [7]:
baseline_fn = results_df.loc[results_df["scenario"] == "Original Traffic", "false_negatives"].values[0]

def compute_increase(fn):
    if baseline_fn == 0:
        return np.nan
    return ((fn - baseline_fn) / baseline_fn) * 100

results_df["fn_percent_increase"] = results_df["false_negatives"].apply(compute_increase)

results_df

,scenario,accuracy,precision,recall,f1,false_negatives,fn_percent_increase
0,Original Traffic,0.999944,0.999980,0.999960,0.999970,2,0.0
1,Timing Perturbation,0.998530,0.999880,0.998537,0.999208,73,3550.0
2,Packet Size Perturbation,0.957252,0.966126,0.988639,0.977253,567,28250.0
3,Packet Rate Perturbation,0.999926,0.999980,0.999940,0.999960,3,50.0
4,Combined Perturbation,0.926191,0.959549,0.961048,0.960298,1944,97100.0


## Save Results

In [6]:
# results_df.to_csv("robustness_results.csv", index=False) // uncomment to save
print("Saved robustness_results.csv")

Saved robustness_results.csv
